In [ ]:
# # 1. 开启 TCP 转发
# !sed -i 's/^AllowTcpForwarding no/AllowTcpForwarding yes/' /tmp/amd-oneclick-sshd_config

# # 2. 热重载 sshd（HUP 信号，不断开现有会话）
# !kill -HUP $(cat /tmp/amd-oneclick-sshd.pid)

In [1]:
!pip install modelscope 

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git

Cloning into 'ComfyUI'...
remote: Enumerating objects: 1442, done.
remote: Counting objects: 100% (1442/1442), done.
remote: Compressing objects: 100% (1243/1243), done.
remote: Total 1442 (delta 157), reused 1173 (delta 154), pack-reused 0 (from 0)
Receiving objects: 100% (1442/1442), 12.34 MiB | 16.71 MiB/s, done.
Resolving deltas: 100% (157/157), done.


In [13]:
!pip install -i https://mirrors.cloud.tencent.com/pypi/simple -r ./ComfyUI/requirements.txt 

Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 69.4 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 74.7 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 41.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 68.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 68.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 93.0 MB/s  0:00:01:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 82.9 MB/s  0:00:01:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 MB 78.5 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 MB 82.5 MB/s  0:00:01m0:00:0100:01
     ━━━━━━

In [11]:
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager.git ./ComfyUI/custom_nodes/ComfyUI-Manager
!pip install -r ./ComfyUI/custom_nodes/ComfyUI-Manager/requirements.txt

fatal: destination path './ComfyUI/custom_nodes/ComfyUI-Manager' already exists and is not an empty directory.
Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [8]:
!export TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1
!export HIP_FORCE_DEV_KERN_ARG=1
!export MIOPEN_FIND_MODE=FAST


In [10]:
import os
import shutil

# 模型下载
from modelscope import snapshot_download

model_dir = snapshot_download('circlestone-labs/Anima')
print("ModelScope 下载完成，模型路径:", model_dir)

# ComfyUI 根目录和模型目录
COMFYUI_ROOT = '/workspace/ComfyUI'
MODELS_ROOT = os.path.join(COMFYUI_ROOT, 'models')

# 映射关系：源文件相对路径 -> ComfyUI 模型子目录
MODELS = [
    # diffusion 模型 -> models/diffusion_models（也可以改成 unet，看你的工作流习惯）
    ('split_files/diffusion_models/anima-turbo-v1.1.safetensors',      'diffusion_models'),
    ('split_files/diffusion_models/anima-base-v1.0.safetensors',       'diffusion_models'),
    ('split_files/diffusion_models/anima-aesthetic-v1.1.safetensors',  'diffusion_models'),
    # 文本编码器 -> models/text_encoders
    ('split_files/text_encoders/qwen_3_06b_base.safetensors',          'text_encoders'),
    # VAE -> models/vae
    ('split_files/vae/qwen_image_vae.safetensors',                     'vae'),
]

for rel, sub in MODELS:
    src = os.path.join(model_dir, rel)
    dst_dir = os.path.join(MODELS_ROOT, sub)
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(rel))

    if os.path.exists(dst):
        print('已有', os.path.basename(rel))
        continue

    if os.path.exists(src):
        try:
            os.symlink(src, dst)
            print(f'已创建软链接: {os.path.basename(rel)} -> {dst}')
        except Exception:
            shutil.copy2(src, dst)
            print(f'已复制文件: {os.path.basename(rel)} -> {dst}')
    else:
        print('警告: 未找到源文件', src)

# 列出各目标目录内容（代替原来的 !ls）
for sub in ['diffusion_models', 'text_encoders', 'vae']:
    target = os.path.join(MODELS_ROOT, sub)
    print(f'\n===== {target} =====')
    if os.path.isdir(target):
        for f in os.listdir(target):
            print(f)
    else:
        print('目录不存在')

2026-10-13 09:11:18,691 | INFO    | modelscope_hub.download | Downloading 18 files from circlestone-labs/Anima@master


Downloading:   0%|          | 0/18 [00:00<?, ?file/s]

ModelScope 下载完成，模型路径: /root/.cache/modelscope/models/circlestone-labs--Anima/snapshots/master
已创建软链接: anima-turbo-v1.1.safetensors -> /workspace/ComfyUI/models/diffusion_models/anima-turbo-v1.1.safetensors
已创建软链接: anima-base-v1.0.safetensors -> /workspace/ComfyUI/models/diffusion_models/anima-base-v1.0.safetensors
已创建软链接: anima-aesthetic-v1.1.safetensors -> /workspace/ComfyUI/models/diffusion_models/anima-aesthetic-v1.1.safetensors
已创建软链接: qwen_3_06b_base.safetensors -> /workspace/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors
已创建软链接: qwen_image_vae.safetensors -> /workspace/ComfyUI/models/vae/qwen_image_vae.safetensors

===== /workspace/ComfyUI/models/diffusion_models =====
anima-base-v1.0.safetensors
anima-turbo-v1.1.safetensors
anima-aesthetic-v1.1.safetensors
put_diffusion_model_files_here

===== /workspace/ComfyUI/models/text_encoders =====
qwen_3_06b_base.safetensors
put_text_encoder_files_here

===== /workspace/ComfyUI/models/vae =====
put_vae_here
qwen_image_vae.safet

In [ ]:
!python ./ComfyUI/main.py --listen 0.0.0.0 --port 8188 --use-pytorch-cross-attention --disable-xformers

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-10-13 09:16:40.508
** Platform: Linux
** Python version: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
** Python executable: /opt/venv/bin/python
** ComfyUI Path: /workspace/ComfyUI
** ComfyUI Base Folder Path: /workspace/ComfyUI
** User directory: /workspace/ComfyUI/user
** ComfyUI-Manager config path: /workspace/ComfyUI/user/__manager/config.ini
** Log path: /workspace/ComfyUI/user/comfyui.log
[INFO] 
Prestartup times for custom nodes:
[INFO]    1.8 seconds: /workspace/ComfyUI/custom_nodes/ComfyUI-Manager
[INFO] 
